# Lab 02 — Sampling, aliasing & clock drift

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. Instructor solutions are not distributed in this repository.

**Covers.** Chapter 2 — §2.7–§2.9 (Nyquist, aliasing, quantization), §2.13 (non-uniform sampling & resampling).

**Biomedical question.** Is my sampling rate and timing good enough for the question I am asking?
**Task type (§2.2, "The measurement pipeline").** Measurement/Monitoring — acquisition (get the samples right before any analysis)
**Information that must be preserved.** the true frequency content (no aliases) *and* the true timing across channels
**Main assumptions.** each channel is labelled with its real sampling rate and the channels share a clock
**Primary diagnostic.** predict the fold-down frequency `f_alias = |f - k*fs|` and confirm it against the FFT peak; track a shared event's cross-channel timing offset over the whole record
**Transfer challenge.** repeat with your own tone / rates and a different drift sign

*Self-contained: seeded synthetic signals only (`rng = np.random.default_rng(2013)`), no file or network I/O, runs in a few seconds.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab02_sampling_aliasing_clock_drift/lab02_sampling_aliasing_clock_drift.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab02_sampling_aliasing_clock_drift.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab02_sampling_aliasing_clock_drift.ipynb)

In [1]:
# --- shared setup (reproducible; fully offline) ---
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

def spectrum(x, fs):
    """One-sided magnitude spectrum; DC zeroed so it never wins argmax."""
    X = np.abs(np.fft.rfft(x)); f = np.fft.rfftfreq(x.size, 1/fs); X[0] = 0.0
    return f, X

def fft_peak_hz(x, fs):
    """Frequency (Hz) of the dominant FFT bin."""
    f, X = spectrum(x, fs)
    return float(f[np.argmax(X)])

def quantize(x, bits, lo=-1.0, hi=1.0):
    """Uniform quantizer over [lo, hi] with 2**bits codes (mid-rise: 0 is not a code)."""
    x = np.clip(x, lo, hi); levels = 2**bits - 1
    return np.round((x - lo)/(hi - lo)*levels)/levels*(hi - lo) + lo

def detect_time(y, t, t_expect, coarse=0.25, sigma=0.004):
    """Sub-sample event time: coarse argmax near t_expect, then an
    amplitude-weighted centroid in a +/-3 sigma window around the peak."""
    win = (t >= t_expect - coarse) & (t <= t_expect + coarse)
    tp = t[win][np.argmax(y[win])]
    fine = (t >= tp - 3*sigma) & (t <= tp + 3*sigma)
    w = np.clip(y[fine], 0, None)
    return float(np.sum(t[fine]*w)/np.sum(w))

## 1. See the alias — predict it, then confirm it
A pure **40 Hz** tone is sampled *below* Nyquist. The fold-down frequency is `f_alias = |f - k*fs|` for the integer `k` that lands it in `[0, fs/2]`. Predict it, then read the FFT peak of the sampled signal and check they agree. (`T = 4 s` is chosen so each alias falls exactly on an FFT bin.)

In [2]:
# TODO under-sample a 40 Hz tone; predict f_alias = |f - k*fs| for the integer k giving the
#      smallest positive value, and CONFIRM it against the FFT peak. Do fs = 70 Hz and 45 Hz.
f0 = 40.0            # true tone (Hz)
T1 = 4.0             # seconds -> FFT bin width 1/T1 = 0.25 Hz
alias = {}           # fs -> (predicted, measured)
for fs in (70.0, 45.0):
    raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: a finer FFT sharpens the peak but does NOT move it back to 40 Hz -- why?

NotImplementedError: TODO: implement this — see the comment above

## 2. More bits do NOT fix aliasing
A common wrong reflex is to buy a higher-resolution ADC. Quantize the *already-aliased* signal at **8** and **16** bits: extra bits only lower the noise floor — the alias peak does not move. Only a higher `fs` or an anti-alias filter removes it.

In [3]:
# TODO quantize the aliased fs=70 Hz signal at 8 vs 16 bits; show the alias peak stays put.
#      For each `bits`: xq = quantize(x_alias, bits); store bit_peak[bits] = fft_peak_hz(xq, fs);
#      then plot 20*np.log10(X/X.max()) (X from spectrum(xq, fs)) for both bit depths on one
#      axis — a linear-magnitude plot will NOT show the noise-floor difference, only the dB plot
#      will.
#      Deliverable: both peaks sit at 30.0 Hz; the noise floor is ~48 dB lower at 16 bit than
#      at 8 bit.
fs = 70.0
n = np.arange(int(round(T1*fs)))
x_alias = np.sin(2*np.pi*f0*n/fs)            # sampled tone -> already folded to 30 Hz
bit_peak = {}
for bits in (8, 16):
    raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: name the ONE change (not more bits) that would remove the 30 Hz alias.

NotImplementedError: TODO: implement this — see the comment above

## 3. Clock drift — a fixed resample cannot fix it
Two channels record the same event train. Channel **B**'s clock runs fast by **0.1%** but is *labelled* at the nominal rate — the analyst never sees the true clock, only B's timestamps at `fs_nom`. We resample the *labelled* B onto the common nominal grid using its own (labelled) timestamps (the honest thing an analyst does) and detect each shared event. The B-vs-A timing offset still grows **linearly** with time: a per-sample rate error is not a constant shift, so no fixed resample removes it.

In [4]:
# TODO channel B's TRUE sample rate is fs_nom*(1+drift), but B is LABELLED (and stored) as if
#      sampled at fs_nom -- the analyst never sees the true clock, only B's labelled timestamps.
#
#      1) Generate y_B by evaluating the pulse train at its TRUE sample instants
#         n/(fs_nom*(1+drift)), but store/label those samples as if taken at n/fs_nom
#         (i.e. index n <-> labelled time n/fs_nom).
#      2) Generate y_A the same way for channel A, with no drift (true == labelled time n/fs_nom).
#      3) Resample the LABELLED B onto A's grid: np.interp(t_A, n/fs_nom, y_B).
#      4) Detect each shared event on both series with detect_time(...) and, IN EVENT ORDER,
#         store offsets_ms = [(tB - tA)*1e3 for tA, tB in zip(times_A, times_B)]  (sign: B - A).
fs_nom  = 500.0
T3      = 60.0
drift   = 0.001                       # +0.1 % clock error on channel B
sigma   = 0.004                       # shared-event pulse width (s)
t_events = np.arange(1.0, T3, 2.0)    # shared events at true times 1, 3, ..., 59 s
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the offset line is a rising ramp, not a flat line -- a single constant shift cannot
#             cancel it. What acquisition-time change (sensor/protocol) actually fixes drift?

NotImplementedError: TODO: implement this — see the comment above

### Live sanity check
The predicted alias must equal the measured FFT peak (to within one bin) at both rates; more bits must not move it; and the drift must accumulate. All numbers are read from the cells above — nothing is hard-coded.

In [5]:
# --- live sanity check (values computed above, not hard-coded) ---
bin_hz = 1.0/T1                       # FFT bin width in section 1
for fs_, (fp, fm) in alias.items():
    assert abs(fp - fm) <= bin_hz, f"alias mismatch at fs={fs_}: predicted {fp} vs measured {fm}"
assert bit_peak[8] == bit_peak[16], "quantization moved the alias -- impossible"
assert abs(offsets_ms[-1] - offsets_ms[0]) > 5, "drift did not accumulate"
print("OK: predicted alias == measured (both rates); extra bits do not move it; drift accumulates.")

KeyError: 8

## Reflection

1. **Stable vs changed.** Across sections 1-2, which conclusion **stayed stable** (the *frequency* of the peak) and which **changed** (the *noise floor*) when you added bits? Explain in one sentence why more amplitude resolution can never undo an under-Nyquist sampling choice.
2. **Evidence for a real recording.** For an actual recording (not this synthetic one), what evidence would convince you there are **no hidden aliases** and **no cross-channel timing drift**? Name the acquisition-side check for each — e.g. a guard band / anti-alias-filter margin below `fs/2`, and a shared hardware clock or a common event marker.
3. **Timing budget.** Suppose your task needs cross-channel timing good to 5 ms. From the measured drift slope, at what record length does a 0.1 % clock error blow that budget?

**Rule out.** Sampling at the *bare* Nyquist rate with no guard band — or assuming a fixed resample can fix a per-sample clock drift — is ruled out: the first lets any content near `fs/2` fold into the band, and the second lets the offset grow without bound. Both break the **§2.2** measurement-pipeline requirement that the samples **preserve the true frequency content and the true cross-channel timing** the downstream task depends on.

> *Your answers here.*